In [3]:
import pandas as pd 
import numpy as np

In [4]:
movies = pd.read_csv("tmdb_5000_movies.csv.zip")
credit = pd.read_csv("tmdb_5000_credits.csv.zip")


In [5]:
movies = movies.merge(credit,on="title")

In [6]:
movies =  movies[["genres", "movie_id",  "overview", "title", "crew","cast", "keywords"]]

In [7]:
movies.dropna(inplace=True)

In [8]:
import ast 

In [9]:
ast.literal_eval

<function ast.literal_eval(node_or_string)>

In [10]:
def convert(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i["name"])
    return L

In [11]:
 movies["genres"] = movies["genres"].apply(convert) 

In [12]:
movies["keywords"]= movies["keywords"].apply(convert) 

In [13]:
def convert3(obj):
    L = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3: 
           L.append(i["name"])
           counter+=1
        else: 
          break
    return L

In [14]:
movies["cast"] =  movies["cast"].apply(convert3) 

In [15]:
def fetch_director(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i["job"]== "Director": 
            L.append(i["name"])
            break
    return L

In [16]:
movies["crew"] =  movies["crew"].apply(fetch_director)

In [17]:
movies["overview"] = movies["overview"].apply(lambda x:x.split())

In [18]:
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast'] = movies["cast"].apply(lambda x:[i.replace(" ","") for i in x])

In [19]:
movies["tags"] = movies["keywords"] + movies["overview"]+ movies["cast"]+  movies["crew"] + movies["genres"]

In [20]:
new_df = movies[["movie_id","title", "tags" ]]


In [21]:
new_df["tags"] = new_df["tags"].apply(lambda x:" ".join(x))

C:\Users\moham\AppData\Local\Temp\ipykernel_12072\1578042357.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df["tags"] = new_df["tags"].apply(lambda x:" ".join(x))


In [22]:
new_df["tags"] = new_df["tags"].apply(lambda x:x.lower())

C:\Users\moham\AppData\Local\Temp\ipykernel_12072\3481706753.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df["tags"] = new_df["tags"].apply(lambda x:x.lower())


In [23]:
import nltk
from nltk.stem.porter import PorterStemmer


In [24]:
ps = PorterStemmer()

In [25]:
def stem(text):
    Y = []
     
    for i in text.split():
      Y.append(ps.stem(i))
    return " ".join(Y)

In [26]:
new_df["tags"] =  new_df["tags"].apply(stem)

C:\Users\moham\AppData\Local\Temp\ipykernel_12072\1074810775.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df["tags"] =  new_df["tags"].apply(stem)


In [27]:
new_df["tags"][0]

'cultureclash futur spacewar spacecoloni societi spacetravel futurist romanc space alien tribe alienplanet cgi marin soldier battl loveaffair antiwar powerrel mindandsoul 3d in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization. samworthington zoesaldana sigourneyweav jamescameron action adventur fantasi sciencefict'

new_df[]

In [28]:
from sklearn.feature_extraction.text import CountVectorizer

In [29]:
cv = CountVectorizer(max_features=5000,stop_words="english")

In [30]:
vectors = cv.fit_transform(new_df["tags"]).toarray()


In [31]:
from sklearn.metrics.pairwise import cosine_similarity

In [32]:
cs = cosine_similarity(vectors)

In [33]:
cs.shape

(4806, 4806)

In [34]:
new_df.head(1)

,movie_id,title,tags
0,19995,Avatar,cultureclash futur spacewar spacecoloni societ...


In [51]:
def recommendation(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    distance = cs[movie_index]

    movie_list = sorted(
        list(enumerate(distance)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    for i in movie_list:
        print(new_df.iloc[i[0]].title)


In [52]:
recommendation('The Dark Knight Rises')

The Dark Knight
Batman Returns
Batman
Batman Forever
Batman Begins


In [53]:
import pickle

In [56]:
pickle.dump(new_df,open('movie_dict.pkl','wb'))
pickle.dump(cs,open('similarity.pkl','wb'))

In [57]:
new_df.head()

,movie_id,title,tags
0,19995,Avatar,cultureclash futur spacewar spacecoloni societ...
1,285,Pirates of the Caribbean: At World's End,ocean drugabus exoticisland eastindiatradingco...
2,206647,Spectre,spi basedonnovel secretag sequel mi6 britishse...
3,49026,The Dark Knight Rises,dccomic crimefight terrorist secretident burgl...
4,49529,John Carter,basedonnovel mar medallion spacetravel princes...
